# Figures for the README, drawn from `outputs/*.csv` only

Run `promotion_ab_test.ipynb` first. This notebook reads result CSVs from `outputs/` and writes PNGs to `images/`.
No result number is typed in here; every number in a figure, label or caption comes from a CSV.

- `TRANSPARENT = False`: white background, saved to `images/`.
- `TRANSPARENT = True`: transparent background for the portfolio site, saved to `images/transparent/`.

In [1]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker
from matplotlib import font_manager
from matplotlib.colors import to_rgb
from matplotlib.patches import Rectangle

from config import OUT_DIR, FIG_DIR

TRANSPARENT = False   # True: transparent PNGs for the site -> images/transparent/
DPI = 200
SAVE_DIR = FIG_DIR / "transparent" if TRANSPARENT else FIG_DIR
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# palette (CLAUDE.md)
INK, INK2, INK3 = "#23233A", "#4A4A66", "#7B7B96"
VIOLET, ROSE, BLUE, MINT = "#7B6FD8", "#D9536B", "#4C7FE0", "#2F9E88"
GRID = "#E6E6EF"
PROMO_COLOR = {1: BLUE, 2: ROSE, 3: MINT}      # Promotion 2 is the one under discussion
SIZES = ["Small", "Medium", "Large"]

def tint(color, a):
    return tuple(1 - (1 - np.array(to_rgb(color))) * a)

_available = {f.name for f in font_manager.fontManager.ttflist}
FONT = "IBM Plex Sans" if "IBM Plex Sans" in _available else "DejaVu Sans"
plt.rcParams.update({
    "font.family": FONT, "font.size": 10,
    "text.color": INK, "axes.labelcolor": INK2, "xtick.color": INK3, "ytick.color": INK3,
    "axes.edgecolor": GRID, "axes.linewidth": 1.0,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.facecolor": "none" if TRANSPARENT else "white",
    "figure.facecolor": "none" if TRANSPARENT else "white",
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.unicode_minus": True,
})
print("font:", FONT, "| transparent:", TRANSPARENT, "| save dir:", SAVE_DIR.name)

def read(name):
    return pd.read_csv(OUT_DIR / f"{name}.csv")

def head(fig, title, subtitle, top=0.80):
    fig.subplots_adjust(top=top)
    fig.canvas.draw()
    r = fig.canvas.get_renderer()
    x0 = min(ax.get_tightbbox(r).x0 for ax in fig.axes)
    x = fig.transFigure.inverted().transform((x0, 0))[0]
    fig.text(x, 0.995, title, ha="left", va="top", fontsize=14, fontweight="bold", color=INK)
    fig.text(x, 0.995 - 0.075, subtitle, ha="left", va="top", fontsize=9, color=INK3, linespacing=1.5)

def save(fig, name):
    fig.savefig(SAVE_DIR / f"{name}.png", dpi=DPI, bbox_inches="tight", pad_inches=0.25,
                transparent=TRANSPARENT, facecolor="none" if TRANSPARENT else "white")
    plt.close(fig)
    print("saved", SAVE_DIR.name + "/" + name + ".png")

def sgn(v, nd=2):
    return f"{v:+.{nd}f}".replace("-", "\u2212")

def neg(s):
    return s.replace("-", "\u2212")

def pv(p):
    return f"= {p:.4f}" if p >= 0.0001 else "< 0.0001"

font: DejaVu Sans | transparent: False | save dir: images


## `qq.png`

In [2]:
qq = read("qq_points"); nm = read("normality").set_index("promotion")
fig, axes = plt.subplots(1, 3, figsize=(11.2, 4.0), sharey=True)
for ax, k in zip(axes, [1, 2, 3]):
    d = qq[qq["promotion"] == k]
    col = PROMO_COLOR[k]
    xs = np.array([d["theoretical_quantile"].min(), d["theoretical_quantile"].max()])
    ax.plot(xs, d["fit_slope"].iloc[0] * xs + d["fit_intercept"].iloc[0], color=INK3, lw=1.4, ls="--", zorder=2)
    ax.scatter(d["theoretical_quantile"], d["ordered_value"], s=26, color=tint(col, 0.55), edgecolor=col, linewidth=0.9, zorder=3)
    ax.set_title(f"Promotion {k}  (n = {int(nm.loc[k, 'n_stores'])} stores)", loc="left", fontsize=10.5, fontweight="bold",
                 color=INK, pad=22)
    ax.text(0, 1.03, f"Shapiro\u2013Wilk W = {nm.loc[k, 'shapiro_W']:.3f}, p {pv(nm.loc[k, 'shapiro_p'])}", transform=ax.transAxes,
            fontsize=8.5, color=INK3)
    ax.set_xlabel("Theoretical normal quantile")
axes[0].set_ylabel("Store four-week average sales ($ thousands)")
head(fig, "Store-level sales depart from normality in the upper tail",
     "Q\u2013Q plots of the store-level four-week averages \u00b7 dashed line = normal fit", top=0.72)
save(fig, "qq")

saved images/qq.png


## `outlier507.png`

In [3]:
s507 = read("store507_weekly"); m6 = read("market6_store_means")
others = m6[~m6["is_store_507"]]["mean_sales_thousands"]
mean507 = m6.loc[m6["is_store_507"], "mean_sales_thousands"].iloc[0]
lo_wk = s507.loc[s507["sales_thousands"].idxmin()]
fig, ax = plt.subplots(figsize=(7.4, 4.6))
ax.axhspan(others.min(), others.max(), color=tint(INK3, 0.14), lw=0, zorder=1)
ax.axhline(others.median(), color=INK3, ls="--", lw=1.2, zorder=2)
ax.text(4.42, others.max(), f"{len(others)} other market-6 stores: averages {others.min():.1f}\u2013{others.max():.1f}", ha="right", va="bottom", fontsize=8.5, color=INK3)
ax.text(4.42, others.median(), f"median {others.median():.1f}", ha="right", va="bottom", fontsize=8.5, color=INK3)
ax.plot(s507["week"], s507["sales_thousands"], color=tint(ROSE, 0.55), lw=2.2, marker="o", ms=7, zorder=3)
ax.plot(lo_wk["week"], lo_wk["sales_thousands"], "o", ms=12, color=ROSE, zorder=4)
ax.annotate(f"week {int(lo_wk['week'])}: {lo_wk['sales_thousands']:.2f}", (lo_wk["week"], lo_wk["sales_thousands"]), xytext=(14, -2),
            textcoords="offset points", fontsize=10, fontweight="bold", color=INK, va="center")
ax.axhline(mean507, color=ROSE, lw=1, ls=":", zorder=2)
ax.text(4.44, mean507 - 0.35, f"store 507\naverage {mean507:.1f}", ha="right", va="top", fontsize=8.5, color=ROSE)
ax.set_xticks(s507["week"]); ax.set_xlim(0.5, 4.5)
ax.set_xlabel("Week"); ax.set_ylabel("Weekly sales ($ thousands)")
head(fig, "Store 507 had one unusually low week",
     "Weekly sales of store 507 against the four-week averages of the other stores in the same market (MarketID 6) \u00b7 kept in the analysis", top=0.84)
save(fig, "outlier507")

saved images/outlier507.png


## `box.png`

In [4]:
bs = read("box_stats").set_index("promotion"); bf = read("box_fliers")
fig, ax = plt.subplots(figsize=(6.8, 4.8))
stats = [dict(med=bs.loc[k, "med"], q1=bs.loc[k, "q1"], q3=bs.loc[k, "q3"], whislo=bs.loc[k, "whislo"], whishi=bs.loc[k, "whishi"],
              mean=bs.loc[k, "mean"], label=f"Promotion {k}\n(n = {int(bs.loc[k, 'n_stores'])})",
              fliers=bf.loc[bf["promotion"] == k, "value"].values) for k in [1, 2, 3]]
bp = ax.bxp(stats, positions=[1, 2, 3], widths=0.4, patch_artist=True, showmeans=True,
            medianprops=dict(color=INK, lw=2.2), whiskerprops=dict(color=INK3, lw=1.2), capprops=dict(color=INK3, lw=1.2),
            meanprops=dict(marker="D", markerfacecolor="white", markeredgecolor=INK, markersize=6),
            flierprops=dict(marker="o", markersize=4, markerfacecolor=tint(INK3, 0.5), markeredgecolor="none"))
for patch, k in zip(bp["boxes"], [1, 2, 3]):
    patch.set_facecolor(tint(PROMO_COLOR[k], 0.55 if k == 2 else 0.3)); patch.set_edgecolor(PROMO_COLOR[k]); patch.set_linewidth(1.8 if k == 2 else 1.4)
for k in [1, 2, 3]:
    ax.text(k + 0.24, bs.loc[k, "mean"], f"mean {bs.loc[k, 'mean']:.1f}", va="center", fontsize=8.5, color=PROMO_COLOR[k], fontweight="bold",
            bbox=dict(fc="white", ec="none", alpha=0.9, pad=1.5))
ax.set_xlim(0.5, 3.9)
ax.set_ylabel("Store four-week average sales ($ thousands)")
ax.grid(axis="x", visible=False)
head(fig, "Promotion 2 had the lowest store-level average sales",
     "Distribution of the 137 store averages by promotion \u00b7 box = quartiles, diamond = mean", top=0.84)
save(fig, "box")

saved images/box.png


## `tukey.png`

In [5]:
ad = read("adjusted_differences")
fig, ax = plt.subplots(figsize=(8.0, 4.0))
ax.axvline(0, color=INK, lw=1.4, zorder=2)
y = np.arange(len(ad))[::-1]
for yy, r in zip(y, ad.itertuples()):
    sig = not (r.tukey_before_ci_low <= 0 <= r.tukey_before_ci_high)
    col = ROSE if sig else tint(INK3, 0.6)
    ax.hlines(yy, r.tukey_before_ci_low, r.tukey_before_ci_high, color=col, lw=3.2 if sig else 2.4, zorder=3)
    ax.plot(r.tukey_before_diff, yy, "o", color=col, ms=10 if sig else 8, zorder=4)
    ax.text(r.tukey_before_ci_high + 0.8, yy, f"{sgn(r.tukey_before_diff)}  (p = {r.tukey_before_p:.4f})", va="center", fontsize=9.5,
            fontweight="bold" if sig else "normal", color=INK if sig else INK3)
ax.set_yticks(y); ax.set_yticklabels([c.replace("-", " \u2212 ").replace("P", "Promotion ") for c in ad["comparison"]])
ax.set_xlim(ad["tukey_before_ci_low"].min() - 3, ad["tukey_before_ci_high"].max() + 11)
ax.set_xlabel("Difference in mean weekly sales per store ($ thousands)")
ax.grid(axis="y", visible=False)
head(fig, "Promotion 2 is below both alternatives; 1 and 3 are not separated",
     "Tukey HSD 95% intervals on the 137 store averages, before adjusting for market size \u00b7 dark = interval excludes zero", top=0.82)
save(fig, "tukey")

saved images/tukey.png


## `segbar.png`

In [6]:
sg = read("segment_means")
piv = sg.pivot(index="market_size", columns="promotion", values="mean_sales_thousands").loc[SIZES]
npv = sg.pivot(index="market_size", columns="promotion", values="n_stores").loc[SIZES]
p2_lowest_everywhere = bool((piv.idxmin(axis=1) == 2).all())
fig, ax = plt.subplots(figsize=(8.4, 4.8))
w = 0.26
for k in [1, 2, 3]:
    xs = np.arange(3) + (k - 2) * w
    ax.bar(xs, piv[k].values, width=w - 0.02, color=PROMO_COLOR[k] if k == 2 else tint(PROMO_COLOR[k], 0.5), label=f"Promotion {k}")
    for x_, v_, n_ in zip(xs, piv[k].values, npv[k].values):
        ax.text(x_, v_ + 1.0, f"{v_:.1f}", ha="center", fontsize=8.8, fontweight="bold" if k == 2 else "normal", color=INK2)
        ax.text(x_, 1.2, f"n={int(n_)}", ha="center", va="bottom", fontsize=7.5, color="white" if k == 2 else INK2)
ax.set_xticks(range(3)); ax.set_xticklabels([f"{s} market" for s in SIZES])
ax.set_ylabel("Mean store sales ($ thousands)")
ax.legend(frameon=False, loc="upper left", fontsize=9)
ax.grid(axis="x", visible=False)
head(fig, "Promotion 2 is lowest in every market size" if p2_lowest_everywhere else "Mean sales by promotion and market size",
     "Store four-week averages by market size \u00b7 market size also differs strongly in the level of sales", top=0.84)
save(fig, "segbar")

saved images/segbar.png


## `promo-means-ci.png`

In [7]:
pm = read("promo_means_ci").set_index("promotion")
fig, ax = plt.subplots(figsize=(7.8, 4.8))
off = 0.17
for k in [1, 2, 3]:
    col = PROMO_COLOR[k]
    ax.vlines(k - off, pm.loc[k, "ci_low_raw"], pm.loc[k, "ci_high_raw"], color=tint(col, 0.5), lw=3.4, zorder=3)
    ax.plot(k - off, pm.loc[k, "mean_raw"], "o", color=tint(col, 0.75), ms=10, zorder=4)
    ax.vlines(k + off, pm.loc[k, "ci_low_adjusted"], pm.loc[k, "ci_high_adjusted"], color=col, lw=3.4, zorder=3)
    ax.plot(k + off, pm.loc[k, "mean_adjusted"], "D", color=col, ms=9, zorder=4)
    ax.text(k - off - 0.07, pm.loc[k, "mean_raw"], f"{pm.loc[k, 'mean_raw']:.1f}", ha="right", va="center", fontsize=9, color=INK2)
    ax.text(k + off + 0.07, pm.loc[k, "mean_adjusted"], f"{pm.loc[k, 'mean_adjusted']:.1f}", ha="left", va="center", fontsize=9,
            fontweight="bold", color=INK)
ax.set_xticks([1, 2, 3]); ax.set_xticklabels([f"Promotion {k}\n({int(pm.loc[k, 'n_stores'])} stores)" for k in [1, 2, 3]])
ax.set_xlim(0.4, 3.6)
ax.set_ylabel("Mean weekly sales per store ($ thousands)")
ax.grid(axis="x", visible=False)
ax.legend(handles=[plt.Line2D([], [], marker="o", color=tint(INK3, 0.6), lw=3, ms=8, label="Raw store-level mean, 95% CI"),
                   plt.Line2D([], [], marker="D", color=INK2, lw=3, ms=7, label="Market-size-adjusted mean, 95% CI")],
          frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2, fontsize=9)
head(fig, "Average store sales by promotion, raw and adjusted for market size",
     "Adjusted means average the three market sizes with equal weight (main-effects model) \u00b7 95% t-intervals, unadjusted for multiple comparisons", top=0.84)
save(fig, "promo-means-ci")

saved images/promo-means-ci.png


## `adjusted-diff.png`

In [8]:
ad = read("adjusted_differences")
fig, ax = plt.subplots(figsize=(8.8, 4.8))
ax.axvline(0, color=INK, lw=1.4, zorder=2)
y0 = np.arange(len(ad))[::-1] * 1.0
dy = 0.17
for yy, r in zip(y0, ad.itertuples()):
    sig_b = not (r.tukey_before_ci_low <= 0 <= r.tukey_before_ci_high)
    sig_a = not (r.adjusted_ci_low_tukey <= 0 <= r.adjusted_ci_high_tukey)
    cb = tint(INK3, 0.75) if sig_b else tint(INK3, 0.45)
    ca = ROSE if sig_a else tint(ROSE, 0.35)
    ax.hlines(yy + dy, r.tukey_before_ci_low, r.tukey_before_ci_high, color=cb, lw=3, zorder=3)
    ax.plot(r.tukey_before_diff, yy + dy, "o", color=cb, ms=8, zorder=4)
    ax.hlines(yy - dy, r.adjusted_ci_low_tukey, r.adjusted_ci_high_tukey, color=ca, lw=3.4, zorder=3)
    ax.plot(r.adjusted_diff, yy - dy, "D", color=ca, ms=8, zorder=4)
    ax.text(max(r.tukey_before_ci_high, r.adjusted_ci_high_tukey) + 0.8, yy + dy, f"before {sgn(r.tukey_before_diff)}", va="center", fontsize=8.8, color=INK3)
    ax.text(max(r.tukey_before_ci_high, r.adjusted_ci_high_tukey) + 0.8, yy - dy, f"adjusted {sgn(r.adjusted_diff)}", va="center", fontsize=8.8,
            fontweight="bold", color=INK)
ax.set_yticks(y0); ax.set_yticklabels([c.replace("-", " \u2212 ").replace("P", "Promotion ") for c in ad["comparison"]])
ax.set_xlim(min(ad["tukey_before_ci_low"].min(), ad["adjusted_ci_low_tukey"].min()) - 3, max(ad["tukey_before_ci_high"].max(), ad["adjusted_ci_high_tukey"].max()) + 9)
ax.set_xlabel("Difference in mean weekly sales per store ($ thousands)")
ax.grid(axis="y", visible=False)
ax.legend(handles=[plt.Line2D([], [], marker="o", color=tint(INK3, 0.75), lw=3, ms=7, label="Before: Tukey HSD on store averages"),
                   plt.Line2D([], [], marker="D", color=ROSE, lw=3, ms=7, label="After: adjusted for market size (Tukey\u2013Kramer)")],
          frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2, fontsize=9)
head(fig, "Adjusting for market size does not remove the gap to Promotion 2",
     "95% intervals, both adjusted for the three pairwise comparisons \u00b7 dark = interval excludes zero", top=0.84)
save(fig, "adjusted-diff")

saved images/adjusted-diff.png


## `business-impact.png`

In [9]:
bi = read("business_scenario")
fig, ax = plt.subplots(figsize=(7.2, 5.0))
for i, r in enumerate(bi.itertuples()):
    col = ROSE if i == 0 else tint(ROSE, 0.55)
    ax.bar(i, r.four_week_total_thousands, width=0.5, color=tint(col, 0.35), edgecolor=col, linewidth=1.6, hatch="//")
    ax.text(i, r.four_week_total_thousands + 40, f"${r.four_week_total_thousands:,.0f}K", ha="center", fontsize=12, fontweight="bold", color=INK)
    ax.text(i, 60, f"{r.weekly_diff_per_store_thousands:.2f}K per store-week\n\u00d7 {int(r.n_promotion2_stores)} stores \u00d7 {int(r.weeks)} weeks",
            ha="center", va="bottom", fontsize=8.5, color=INK2, bbox=dict(fc="white", ec="none", alpha=0.85, pad=2))
ax.set_xticks(range(len(bi))); ax.set_xticklabels([r.reference for r in bi.itertuples()])
ax.set_ylim(0, bi["four_week_total_thousands"].max() * 1.22)
ax.set_ylabel("Four-week scenario amount ($ thousands)")
ax.grid(axis="x", visible=False)
ax.text(0.98, 0.97, "SCENARIO ESTIMATE", transform=ax.transAxes, ha="right", va="top", fontsize=11, fontweight="bold", color=ROSE)
head(fig, "What the Promotion 2 gap would amount to across the 47 stores",
     "Scenario estimate: the Tukey differences applied to every Promotion 2 store for four weeks. Not a measured loss; ignores costs, margins and store variation.", top=0.80)
save(fig, "business-impact")

saved images/business-impact.png
